# Data Objects & Access Control

This final notebook ties together the full picture of data objects in Databricks — tables, views, functions, volumes — and how access control works across them in a Unity Catalog environment. It also covers Delta Sharing and the practical patterns you need for the exam.

Topics:
- Managed vs external tables (revisited from a governance perspective)
- Views and their access control implications
- User-defined functions as securables
- Volumes for file access
- Row filters and column masks (native UC features)
- Delta Sharing: sharing data across organisations
- Information schema — discovering objects and grants programmatically
- Common exam patterns: who can do what

## 1. Managed vs External Tables — Governance Perspective

The distinction between managed and external tables is critical for understanding what happens to data when access is revoked or an object is dropped.

### Managed Table
- Data files stored **inside** the UC-managed storage path
- Unity Catalog fully controls the data lifecycle
- `DROP TABLE` → **data files are deleted**
- `DROP SCHEMA CASCADE` → all tables and their data are deleted
- `DROP CATALOG CASCADE` → everything inside is deleted

### External Table
- Data files stored **outside** UC-managed storage, at an External Location path
- UC governs access (SELECT, MODIFY) but does **not** own the files
- `DROP TABLE` → table metadata deleted, **data files remain**
- Safe to drop and re-register without losing data

**Exam tip:** Managed = UC owns data. External = UC governs access only. Dropping a managed table deletes the data; dropping an external table does not.

In [ ]:
# Setup
spark.sql("CREATE CATALOG IF NOT EXISTS demo_gov")
spark.sql("CREATE SCHEMA IF NOT EXISTS demo_gov.retail")

# Managed table — data stored inside UC managed storage
spark.sql("""
    CREATE TABLE IF NOT EXISTS demo_gov.retail.customers (
        customer_id   BIGINT,
        email         STRING,
        region        STRING,
        account_tier  STRING
    )
""")

spark.sql("""
    INSERT INTO demo_gov.retail.customers VALUES
    (1, 'alice@example.com', 'EMEA', 'premium'),
    (2, 'bob@example.com',   'APAC', 'standard'),
    (3, 'carol@example.com', 'EMEA', 'premium'),
    (4, 'dave@example.com',  'AMER', 'standard')
""")

# DESCRIBE EXTENDED shows whether a table is MANAGED or EXTERNAL
spark.sql("DESCRIBE EXTENDED demo_gov.retail.customers").show(truncate=False)

## 2. Views as an Access Control Mechanism

Views are securables — you can GRANT SELECT on a view without granting SELECT on the underlying table. This is the primary mechanism for exposing a safe subset of data to consumers.

### View ownership and access
- A view's owner must have SELECT on all underlying tables
- Users granted SELECT on the view do **not** need SELECT on the underlying tables
- If the underlying table's permissions change such that the view owner loses access, the view breaks for all consumers

**Pattern:** Use a service principal as the view owner so that ownership is not tied to an individual user who might leave the organisation.

In [ ]:
# View that restricts columns — no email exposed
spark.sql("""
    CREATE OR REPLACE VIEW demo_gov.retail.customers_public AS
    SELECT customer_id, region, account_tier
    FROM demo_gov.retail.customers
""")

# Consumers with SELECT on customers_public cannot see email
# even though email exists in the underlying table
spark.sql("SELECT * FROM demo_gov.retail.customers_public").show()

In [ ]:
# Dynamic view — row filter using current_user()
# Each user only sees rows for their region
# Admins see everything
spark.sql("""
    CREATE OR REPLACE VIEW demo_gov.retail.customers_by_region AS
    SELECT * FROM demo_gov.retail.customers
    WHERE
        is_account_group_member('data-admins')
        OR region = (
            SELECT region
            FROM demo_gov.retail.customers
            WHERE email = current_user()
            LIMIT 1
        )
""")

## 3. Row Filters and Column Masks (Native UC)

Unity Catalog supports **row filters** and **column masks** as first-class table properties — separate from views. They are declared as SQL functions and attached to the table.

### Row Filter
A function that returns a boolean. Attached to a table with `ALTER TABLE ... SET ROW FILTER`. Every query against the table automatically applies the filter. Consumers cannot bypass it.

### Column Mask
A function that returns a masked value for a column. Attached with `ALTER TABLE ... ALTER COLUMN ... SET MASK`. Every query that selects that column receives the masked value.

Unlike dynamic views, row filters and column masks are invisible to consumers — the table appears normal, but data is filtered/masked transparently.

In [ ]:
# Define a row filter function
spark.sql("""
    CREATE OR REPLACE FUNCTION demo_gov.retail.region_filter(region STRING)
    RETURN
        is_account_group_member('data-admins')
        OR region = current_user()
""")

# Attach row filter to the table
spark.sql("""
    ALTER TABLE demo_gov.retail.customers
    SET ROW FILTER demo_gov.retail.region_filter ON (region)
""")

In [ ]:
# Define a column mask function for the email column
spark.sql("""
    CREATE OR REPLACE FUNCTION demo_gov.retail.mask_email(email STRING)
    RETURN
        CASE
            WHEN is_account_group_member('data-admins') THEN email
            ELSE CONCAT(LEFT(email, 2), '****@****.com')
        END
""")

# Attach mask to the email column
spark.sql("""
    ALTER TABLE demo_gov.retail.customers
    ALTER COLUMN email SET MASK demo_gov.retail.mask_email
""")

# Verify: non-admins see masked email
spark.sql("SELECT * FROM demo_gov.retail.customers").show(truncate=False)

In [ ]:
# Remove a row filter or column mask
spark.sql("ALTER TABLE demo_gov.retail.customers DROP ROW FILTER")
spark.sql("ALTER TABLE demo_gov.retail.customers ALTER COLUMN email DROP MASK")

**Exam tip:** Row filters and column masks are attached to the **table**, not the view. They apply regardless of how the table is accessed — directly, through a view, or from a DLT pipeline. Dynamic views are the older pattern; row filters and column masks are the newer, recommended pattern in Unity Catalog.

## 4. User-Defined Functions as Securables

SQL UDFs registered in Unity Catalog are securables. You GRANT and REVOKE `EXECUTE` privilege on them.

```sql
GRANT EXECUTE ON FUNCTION demo_gov.retail.mask_email TO `analysts`;
REVOKE EXECUTE ON FUNCTION demo_gov.retail.mask_email FROM `analysts`;
SHOW GRANTS ON FUNCTION demo_gov.retail.mask_email;
```

Functions defined in Unity Catalog are persistent across sessions and accessible to any user with `EXECUTE` privilege and `USE SCHEMA` on the containing schema.

**Contrast with Python UDFs:** Python UDFs registered with `spark.udf.register()` are session-scoped and not Unity Catalog securables. To share a Python UDF across users, package it in a wheel and install it on the cluster, or wrap it in a SQL UDF body.

In [ ]:
# UC-registered SQL UDF — persistent, securable
spark.sql("""
    CREATE OR REPLACE FUNCTION demo_gov.retail.tier_label(tier STRING)
    RETURNS STRING
    RETURN CASE
        WHEN tier = 'premium'  THEN 'Gold'
        WHEN tier = 'standard' THEN 'Silver'
        ELSE 'Bronze'
    END
""")

spark.sql("""
    SELECT
        customer_id,
        region,
        demo_gov.retail.tier_label(account_tier) AS tier_label
    FROM demo_gov.retail.customers
""").show()

## 5. Volumes for File Access

Volumes are Unity Catalog objects representing a cloud storage path. They enable governed file access without direct cloud credentials.

### Managed Volume
Storage path is inside UC-managed storage. Files are deleted when the volume is dropped.

### External Volume
Storage path points to an External Location. Dropping the volume does not delete the files.

```sql
-- Managed volume
CREATE VOLUME demo_gov.retail.raw_files;

-- External volume
CREATE EXTERNAL VOLUME demo_gov.retail.external_files
LOCATION 's3://my-bucket/retail/raw/';
```

Access a volume path in Python:
```python
df = spark.read.format("json").load("/Volumes/demo_gov/retail/raw_files/2025/04/")
```

### Volume Privileges
- `READ VOLUME` — list and read files
- `WRITE VOLUME` — create and overwrite files

**Exam tip:** In UC-enabled workspaces, use `/Volumes/...` paths instead of `dbfs:/` for file access. The `dbfs:/` root mount is a legacy mechanism not compatible with UC governance.

## 6. Delta Sharing

Delta Sharing is an open protocol for sharing live Delta Lake data across organisations, clouds, and platforms — without copying the data.

### Key Concepts

| Object | Description |
|--------|-------------|
| **Share** | A named collection of tables (or partitions of tables) you want to share |
| **Recipient** | An external organisation or user who receives access to a share |
| **Provider** | The organisation that owns the data and creates the share |

### Provider side

In [ ]:
# Create a share
spark.sql("CREATE SHARE IF NOT EXISTS retail_partner_share")

# Add a table to the share (can also add a partition or a view)
spark.sql("""
    ALTER SHARE retail_partner_share
    ADD TABLE demo_gov.retail.customers_public
""")

# Create a recipient — generates an activation link or uses an open sharing profile
spark.sql("CREATE RECIPIENT IF NOT EXISTS partner_corp")

# Grant the recipient access to the share
spark.sql("GRANT SELECT ON SHARE retail_partner_share TO RECIPIENT partner_corp")

# Show what is in the share
spark.sql("SHOW ALL IN SHARE retail_partner_share").show(truncate=False)

### Consumer side

The recipient receives a credential file or token. They use it to create a **provider** object in their own Databricks workspace and then read the shared tables:

```sql
-- Consumer workspace
CREATE PROVIDER my_supplier
    USING DATABRICKS
    RECIPIENT_PROFILE_STR '{ ... credential JSON ... }';

SHOW SHARES IN PROVIDER my_supplier;

CREATE CATALOG shared_retail
    USING SHARE my_supplier.retail_partner_share;

SELECT * FROM shared_retail.retail.customers_public;
```

**Exam tip:** Delta Sharing is **read-only** for the recipient. The data stays in the provider's storage; the recipient queries it live through the Delta Sharing protocol. There is no data copy.

## 7. Information Schema

Every Unity Catalog schema has a built-in `information_schema` that lets you programmatically discover objects and grants.

```sql
-- List all tables in a schema
SELECT table_name, table_type, created_by
FROM demo_gov.information_schema.tables
WHERE table_schema = 'retail'
ORDER BY table_name;

-- List all columns in a table
SELECT column_name, data_type, is_nullable
FROM demo_gov.information_schema.columns
WHERE table_schema = 'retail'
  AND table_name   = 'customers'
ORDER BY ordinal_position;

-- List all grants on objects in a catalog
SELECT grantee, privilege_type, object_type, object_name
FROM demo_gov.information_schema.object_privileges
ORDER BY object_type, object_name;
```

**Exam tip:** `information_schema` is catalog-scoped — you query `catalog.information_schema.tables`, not a global table. The `system` catalog's tables (`system.access.audit`, `system.lakeflow.job_runs`) are separate and cover the whole account.

In [ ]:
# Discover objects programmatically
spark.sql("""
    SELECT table_name, table_type
    FROM demo_gov.information_schema.tables
    WHERE table_schema = 'retail'
    ORDER BY table_name
""").show(truncate=False)

spark.sql("""
    SELECT column_name, data_type, is_nullable
    FROM demo_gov.information_schema.columns
    WHERE table_schema = 'retail'
      AND table_name   = 'customers'
    ORDER BY ordinal_position
""").show(truncate=False)

## 8. Common Exam Scenarios

### Scenario A — Minimum privileges to SELECT from a table
```sql
GRANT USE CATALOG ON CATALOG main TO `analysts`;
GRANT USE SCHEMA  ON SCHEMA  main.gold TO `analysts`;
GRANT SELECT      ON TABLE   main.gold.revenue TO `analysts`;
```

### Scenario B — Allow a team to create tables in a schema
```sql
GRANT USE CATALOG  ON CATALOG main TO `engineers`;
GRANT USE SCHEMA   ON SCHEMA  main.bronze TO `engineers`;
GRANT CREATE TABLE ON SCHEMA  main.bronze TO `engineers`;
GRANT MODIFY       ON SCHEMA  main.bronze TO `engineers`;  -- to INSERT into created tables
```

### Scenario C — Share data without exposing PII
Option 1: Create a view that excludes PII columns → GRANT SELECT on the view.
Option 2: Attach a column mask to the PII column → GRANT SELECT on the table.

### Scenario D — Drop a schema and all its data
```sql
DROP SCHEMA main.bronze CASCADE;  -- drops all managed tables + their data files
```
External tables in the schema will have their metadata dropped but data files remain.

### Scenario E — Audit who accessed a table yesterday
```sql
SELECT event_time, user_identity.email, action_name
FROM system.access.audit
WHERE DATE(event_time) = CURRENT_DATE - INTERVAL 1 DAY
  AND request_params.table_full_name = 'main.gold.revenue'
ORDER BY event_time;
```

## 9. Cleanup

In [ ]:
spark.sql("DROP SHARE IF EXISTS retail_partner_share")
spark.sql("DROP RECIPIENT IF EXISTS partner_corp")
spark.sql("DROP CATALOG IF EXISTS demo_gov CASCADE")

## 10. Full Data Governance Summary

| Topic | Key point |
|-------|-----------|
| Managed table | UC owns data — DROP TABLE deletes files |
| External table | UC governs access — DROP TABLE keeps files |
| View | Securable — GRANT SELECT without granting on underlying table |
| Row filter | Function attached to table — filters rows transparently |
| Column mask | Function attached to column — masks value transparently |
| Dynamic view | Older pattern for row/column security via `current_user()` |
| SQL UDF (UC) | Persistent, securable with EXECUTE privilege |
| Volume | Governed file access — use `/Volumes/` not `dbfs:/` |
| Delta Sharing | Live read-only cross-org sharing — no data copy |
| information_schema | Catalog-scoped object and privilege discovery |
| system.access.audit | Account-wide access event log |
| Account groups | Only type usable as UC principals (not workspace-local) |
| Minimum SELECT | USE CATALOG + USE SCHEMA + SELECT |
| No Isolation Shared | Incompatible with UC — use Single User or Shared mode |